# UCI Parkinson Speech (Multiple Recordings)

This notebook trains a speech-only Parkinson's classifier using the 26 acoustic features from the UCI multiple-recordings dataset. The UPDRS score is loaded and inspected, but not used as an input feature for the classifier.

In [9]:
from __future__ import annotations

from pathlib import Path
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.utils import check_random_state


In [ ]:
# Paths
TRAIN_PATH = Path(r"datasets\speech\parkinson+speech+dataset+with+multiple+types+of+sound+recordings\train_data.csv")
TEST_PATH = Path(r"datasets\speech\parkinson+speech+dataset+with+multiple+types+of+sound+recordings\test_data.csv")
ARTIFACT_DIR = Path("artifacts/speech/uci_mul")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Column names based on the data dictionary you provided
TRAIN_COLUMNS = [
    "subject_id",
    "jitter_local",
    "jitter_local_abs",
    "jitter_rap",
    "jitter_ppq5",
    "jitter_ddp",
    "shimmer_local",
    "shimmer_local_db",
    "shimmer_apq3",
    "shimmer_apq5",
    "shimmer_apq11",
    "shimmer_dda",
    "ac",
    "nth",
    "htn",
    "median_pitch",
    "mean_pitch",
    "pitch_std",
    "min_pitch",
    "max_pitch",
    "num_pulses",
    "num_periods",
    "mean_period",
    "period_std",
    "fraction_unvoiced_frames",
    "num_voice_breaks",
    "degree_voice_breaks",
    "updrs",
    "label",
]

TEST_COLUMNS = TRAIN_COLUMNS[:-2] + ["label"]

SPEECH_FEATURES = [
    "jitter_local",
    "jitter_local_abs",
    "jitter_rap",
    "jitter_ppq5",
    "jitter_ddp",
    "shimmer_local",
    "shimmer_local_db",
    "shimmer_apq3",
    "shimmer_apq5",
    "shimmer_apq11",
    "shimmer_dda",
    "ac",
    "nth",
    "htn",
    "median_pitch",
    "mean_pitch",
    "pitch_std",
    "min_pitch",
    "max_pitch",
    "num_pulses",
    "num_periods",
    "mean_period",
    "period_std",
    "fraction_unvoiced_frames",
    "num_voice_breaks",
    "degree_voice_breaks",
]

TARGET_COL = "label"
GROUP_COL = "subject_id"
DROP_COLS = [GROUP_COL, "updrs", TARGET_COL]

In [11]:
# Load data
train_df = pd.read_csv(TRAIN_PATH, header=None, names=TRAIN_COLUMNS)
test_df = pd.read_csv(TEST_PATH, header=None, names=TEST_COLUMNS)

display(train_df.head())
display(test_df.head())

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:", int(train_df.isna().sum().sum()))
print("Missing values in test :", int(test_df.isna().sum().sum()))

print("\nLabel distribution (train):")
print(train_df[TARGET_COL].value_counts().sort_index())

print("\nSubject count (train):", train_df[GROUP_COL].nunique())
print("Subject count (test) :", test_df[GROUP_COL].nunique())

,subject_id,jitter_local,jitter_local_abs,jitter_rap,jitter_ppq5,jitter_ddp,shimmer_local,shimmer_local_db,shimmer_apq3,shimmer_apq5,...,max_pitch,num_pulses,num_periods,mean_period,period_std,fraction_unvoiced_frames,num_voice_breaks,degree_voice_breaks,updrs,label
0,1,1.488,0.000090,0.900,0.794,2.699,8.334,0.779,4.517,4.609,...,187.576,160,159,0.006065,0.000416,0.000,0,0.000,23,1
1,1,0.728,0.000038,0.353,0.376,1.059,5.864,0.642,2.058,3.180,...,234.505,170,169,0.005181,0.000403,2.247,0,0.000,23,1
2,1,1.220,0.000074,0.732,0.670,2.196,8.719,0.875,4.347,5.166,...,211.442,1431,1427,0.006071,0.000474,10.656,1,0.178,23,1
3,1,2.502,0.000123,1.156,1.634,3.469,13.513,1.273,5.263,8.771,...,220.230,94,92,0.004910,0.000320,0.000,0,0.000,23,1
4,1,3.509,0.000167,1.715,1.539,5.145,9.112,1.040,3.102,4.927,...,225.162,117,114,0.004757,0.000380,18.182,1,13.318,23,1


,subject_id,jitter_local,jitter_local_abs,jitter_rap,jitter_ppq5,jitter_ddp,shimmer_local,shimmer_local_db,shimmer_apq3,shimmer_apq5,...,min_pitch,max_pitch,num_pulses,num_periods,mean_period,period_std,fraction_unvoiced_frames,num_voice_breaks,degree_voice_breaks,label
0,1,0.135,0.000007,0.067,0.078,0.202,2.033,0.178,1.074,1.336,...,184.502,187.880,183.0,182.0,0.005368,0.000025,0.0,0.0,0.0,1
1,1,0.143,0.000007,0.073,0.081,0.219,1.236,0.107,0.612,0.904,...,198.665,202.214,194.0,193.0,0.004988,0.000020,0.0,0.0,0.0,1
2,1,0.162,0.000008,0.087,0.089,0.260,1.338,0.117,0.630,0.948,...,197.220,206.060,198.0,197.0,0.004940,0.000046,0.0,0.0,0.0,1
3,1,0.140,0.000007,0.075,0.089,0.224,1.086,0.094,0.556,0.747,...,202.324,206.182,200.0,199.0,0.004900,0.000023,0.0,0.0,0.0,1
4,1,0.150,0.000007,0.080,0.097,0.240,1.049,0.091,0.533,0.698,...,205.407,209.927,204.0,203.0,0.004820,0.000022,0.0,0.0,0.0,1


Train shape: (1040, 29)
Test shape : (168, 28)

Missing values in train: 0
Missing values in test : 0

Label distribution (train):
label
0    520
1    520
Name: count, dtype: int64

Subject count (train): 40
Subject count (test) : 28


In [12]:
# Basic sanity checks
X_full = train_df[SPEECH_FEATURES].copy()
y_full = train_df[TARGET_COL].astype(int).copy()
groups_full = train_df[GROUP_COL].astype(int).copy()

assert X_full.shape[1] == 26, f"Expected 26 speech features, got {X_full.shape[1]}"
assert set(y_full.unique()).issubset({0, 1}), "Labels should be binary (0/1)"

print("Feature matrix shape:", X_full.shape)
print("Target shape:", y_full.shape)
print("Unique labels:", sorted(y_full.unique().tolist()))
print(groups_full.shape)

Feature matrix shape: (1040, 26)
Target shape: (1040,)
Unique labels: [0, 1]
(1040,)


In [13]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        X_full,
        y_full,
        groups=groups_full
    )
)

X_train = X_full.iloc[train_idx]
y_train = y_full.iloc[train_idx]

X_val = X_full.iloc[val_idx]
y_val = y_full.iloc[val_idx]

groups_train = groups_full.iloc[train_idx]
groups_val = groups_full.iloc[val_idx]

print(set(groups_train).intersection(set(groups_val)))

set()


In [14]:
RANDOM_STATE = 42

BASE_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=4,
    min_child_samples=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1
)

In [17]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

best_k = None
best_auc = -1

K_VALUES = [10, 15, 20, 25, 30, 40, 50]

print("Searching best k...\n")

for k in K_VALUES:

    fold_aucs = []

    for tr_idx, cv_idx in cv.split(
        X_train,
        y_train,
        groups=groups_train
    ):

        Xtr = X_train.iloc[tr_idx]
        ytr = y_train.iloc[tr_idx]

        Xcv = X_train.iloc[cv_idx]
        ycv = y_train.iloc[cv_idx]

        selector = SelectKBest(
            score_func=mutual_info_classif,
            k=min(k, X_train.shape[1])
        )

        Xtr_sel = selector.fit_transform(
            Xtr,
            ytr
        )

        Xcv_sel = selector.transform(
            Xcv
        )

        model = LGBMClassifier(
            **BASE_PARAMS
        )

        model.fit(
            Xtr_sel,
            ytr
        )

        probs = model.predict_proba(
            Xcv_sel
        )[:, 1]

        auc = roc_auc_score(
            ycv,
            probs
        )

        fold_aucs.append(auc)

    mean_auc = np.mean(fold_aucs)

    print(
        f"k={k:2d} | CV AUC={mean_auc:.4f}"
    )

    if mean_auc > best_auc:
        best_auc = mean_auc
        best_k = k

print("\n====================")
print(f"Best k       : {best_k}")
print(f"Best CV AUC  : {best_auc:.4f}")
print("====================")

Searching best k...



d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not ha

k=10 | CV AUC=0.5491


d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not ha

k=15 | CV AUC=0.5655


...

d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not ha

k=40 | CV AUC=0.5758


d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


k=50 | CV AUC=0.5758

Best k       : 25
Best CV AUC  : 0.5777


d:\Parkinson\Parkinsons-Multimodal-Framework\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [19]:
model = LGBMClassifier(**BASE_PARAMS)

scores = []

for tr_idx, cv_idx in cv.split(
    X_train,
    y_train,
    groups=groups_train
):

    Xtr = X_train.iloc[tr_idx]
    ytr = y_train.iloc[tr_idx]

    Xcv = X_train.iloc[cv_idx]
    ycv = y_train.iloc[cv_idx]

    model.fit(Xtr, ytr)

    probs = model.predict_proba(Xcv)[:, 1]

    scores.append(
        roc_auc_score(ycv, probs)
    )

print(np.mean(scores))

0.5757725180802103


In [20]:
print(y_full.value_counts())

label
1    520
0    520
Name: count, dtype: int64


In [17]:
# Choose k using group-aware cross-validation on the training split
cv = GroupKFold(n_splits=min(CV_SPLITS, groups_train.nunique()))

k_results = []
for k in K_CANDIDATES:
    selector = SelectKBest(score_func=f_classif, k=min(k, X_train.shape[1]))
    clf = RandomForestClassifier(**BASE_PARAMS)
    pipe = Pipeline([
        ("selector", selector),
        ("clf", clf),
    ])
    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        groups=groups_train,
        scoring="roc_auc",
        n_jobs=-1,
    )
    k_results.append((k, float(scores.mean()), float(scores.std())))
    print(f"k={k:>2} | AUC={scores.mean():.4f} ± {scores.std():.4f}")

best_k, best_auc, best_auc_std = max(k_results, key=lambda t: (t[1], -t[0]))
print("\nBest k:", best_k, "best CV AUC:", best_auc)

k= 8 | AUC=0.6185 ± 0.0752
k=10 | AUC=0.6581 ± 0.0961
k=12 | AUC=0.6468 ± 0.0774
k=14 | AUC=0.6521 ± 0.0768
k=16 | AUC=0.6522 ± 0.0907
k=18 | AUC=0.6501 ± 0.0921
k=20 | AUC=0.6578 ± 0.0962
k=22 | AUC=0.6466 ± 0.1011
k=24 | AUC=0.6464 ± 0.0970
k=26 | AUC=0.6466 ± 0.0940

Best k: 10 best CV AUC: 0.658095003287311


In [18]:
# Fit selector and base model on the training split
selector = SelectKBest(score_func=f_classif, k=min(best_k, X_train.shape[1]))
X_train_sel = selector.fit_transform(X_train, y_train)
X_val_sel = selector.transform(X_val)

base_model = RandomForestClassifier(**BASE_PARAMS)
base_model.fit(X_train_sel, y_train)

feature_cols_full = SPEECH_FEATURES
selected_idx = selector.get_support(indices=True)
selected_feature_names = [feature_cols_full[i] for i in selected_idx]

print("Selected features:", selected_feature_names)
print("Selected matrix shape:", X_train_sel.shape)

Selected features: ['jitter_local', 'jitter_local_abs', 'jitter_rap', 'jitter_ppq5', 'jitter_ddp', 'shimmer_apq11', 'pitch_std', 'max_pitch', 'fraction_unvoiced_frames', 'degree_voice_breaks']
Selected matrix shape: (832, 10)


In [19]:
# Validation evaluation (NO calibration)

val_prob = base_model.predict_proba(X_val_sel)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)

threshold_scores = []

for t in thresholds:

    pred = (val_prob >= t).astype(int)

    threshold_scores.append(
        (
            float(t),
            float(
                balanced_accuracy_score(
                    y_val,
                    pred
                )
            ),
            float(
                f1_score(
                    y_val,
                    pred,
                    zero_division=0
                )
            ),
        )
    )

best_threshold, best_bal_acc, best_f1 = max(
    threshold_scores,
    key=lambda x: (
        x[1],
        x[2],
        -abs(x[0] - 0.5)
    ),
)

validation_auc = roc_auc_score(
    y_val,
    val_prob
)

print("Validation AUC:", validation_auc)
print("Best threshold:", best_threshold)
print("Balanced Accuracy:", best_bal_acc)
print("F1:", best_f1)

Validation AUC: 0.6410256410256412
Best threshold: 0.3999999999999999
Balanced Accuracy: 0.641025641025641
F1: 0.7142857142857143


In [20]:
# Fit a final model on all labeled data for SHAP/inference
# This is the model that inference.py will explain.
selector_full = SelectKBest(score_func=f_classif, k=min(best_k, X_full.shape[1]))
X_full_sel = selector_full.fit_transform(X_full, y_full)
final_model = RandomForestClassifier(**BASE_PARAMS)
final_model.fit(X_full_sel, y_full)

selected_idx_full = selector_full.get_support(indices=True)
selected_feature_names_full = [feature_cols_full[i] for i in selected_idx_full]

# Calibrated probability model trained with internal cross-validation
cal_model_full = CalibratedClassifierCV(
    estimator=RandomForestClassifier(**BASE_PARAMS),
    method="sigmoid",
    cv=GroupKFold(
        n_splits=min(
            CV_SPLITS,
            groups_full.nunique()
        )
    ),
)

cal_model_full.fit(
    X_full_sel,
    y_full,
)

# Subject-aware out-of-fold estimate for threshold selection and model reporting
oof_pipe = Pipeline([
    ("selector", SelectKBest(score_func=f_classif, k=min(best_k, X_full.shape[1]))),
    ("clf", RandomForestClassifier(**BASE_PARAMS)),
])

oof_prob = cross_val_predict(
    oof_pipe,
    X_full,
    y_full,
    groups=groups_full,
    cv=GroupKFold(n_splits=min(CV_SPLITS, groups_full.nunique())),
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

oof_df = pd.DataFrame({
    "subject_id": groups_full,
    "label": y_full,
    "prob": oof_prob
})

subject_oof = (
    oof_df
    .groupby("subject_id")
    .agg({
        "prob": "mean",
        "label": "first"
    })
    .reset_index()
)

validation_auc = roc_auc_score(
    subject_oof["label"],
    subject_oof["prob"]
)

print(
    "Subject-level OOF AUC:",
    validation_auc
)

thresholds = np.linspace(0.05, 0.95, 181)
threshold_scores = []
for t in thresholds:
    pred = (oof_prob >= t).astype(int)
    threshold_scores.append(
        (
            float(t),
            float(balanced_accuracy_score(y_full, pred)),
            float(f1_score(y_full, pred, zero_division=0)),
        )
    )

best_threshold, best_bal_acc, best_f1 = max(
    threshold_scores,
    key=lambda x: (x[1], x[2], -abs(x[0] - 0.5)),
)

print("Final model trained on full data.")
print("Final selected feature count:", len(selected_feature_names_full))
print("OOF validation AUC:", validation_auc)
print("Best threshold from OOF probs:", best_threshold)
print("OOF balanced accuracy:", best_bal_acc)
print("OOF F1:", best_f1)

ValueError: The 'groups' parameter should not be None.

In [ ]:
# Bootstrap subjects, not recordings

rng = np.random.default_rng(42)

unique_subjects = np.unique(
    groups_full
)

n_bootstrap = 25

bootstrap_models = []

for b in range(n_bootstrap):

    sampled_subjects = rng.choice(
        unique_subjects,
        size=len(unique_subjects),
        replace=True
    )

    boot_indices = []

    for subject in sampled_subjects:

        idx = np.where(
            groups_full.values == subject
        )[0]

        boot_indices.extend(
            idx.tolist()
        )

    boot_indices = np.array(
        boot_indices
    )

    boot_model = RandomForestClassifier(
        **BASE_PARAMS
    )

    boot_model.fit(
        X_full_sel[boot_indices],
        y_full.iloc[boot_indices]
    )

    bootstrap_models.append(
        boot_model
    )

print(
    f"Trained {len(bootstrap_models)} bootstrap models."
)

In [ ]:
# Save artifacts
artifact_map = {
    "model": final_model,
    "cal_model": cal_model_full,
    "selector": selector_full,
    "feature_cols_full": feature_cols_full,
    "selected_feature_names": selected_feature_names_full,
    "bootstrap_models": bootstrap_models,
    "validation_auc": float(validation_auc),
    "decision_threshold": float(best_threshold),
}

paths = {
    "model": ARTIFACT_DIR / "speech_uci_mul_rf_v1.pkl",
    "cal_model": ARTIFACT_DIR / "speech_uci_mul_rf_v1_calibrated.pkl",
    "selector": ARTIFACT_DIR / "feature_selector.pkl",
    "feature_cols_full": ARTIFACT_DIR / "feature_cols_full.pkl",
    "selected_feature_names": ARTIFACT_DIR / "selected_feature_names.pkl",
    "bootstrap_models": ARTIFACT_DIR / "bootstrap_models.pkl",
    "validation_auc": ARTIFACT_DIR / "validation_auc.pkl",
    "decision_threshold": ARTIFACT_DIR / "decision_threshold.pkl",
}

for key, obj in artifact_map.items():
    joblib.dump(obj, paths[key])

print("Saved artifacts to:", ARTIFACT_DIR.resolve())

## Notes

- `subject_id` and `updrs` are excluded from the classifier input.
- If you later decide to build a severity regressor, `updrs` can be trained as a separate target.
- The saved artifacts match the runtime files expected by `model.py` and `inference.py`.